In [1]:
import os

# policy
from huggingface_hub import snapshot_download
from lerobot.configs.policies import PreTrainedConfig

# record utils
from lerobot.record import record, RecordConfig, DatasetRecordConfig

# torch
from torch import cuda

# utils
import pprint
from dotenv import load_dotenv

# my code
from robot.robot_config import robot_config
from robot.robot_const import TABLE_START_POSE_OPEN, FOLDED_START_POSE
from src.paths import REPO_ROOT, HF_NAME, POLICIES_DIR, EVAL_DIR
from src.utils import check_resume

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

/home/jonathan/miniforge3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


Set Params:

In [2]:
PUSH_TO_HUB       = False
SAVE_TO_DATASET   = False
REPO_NAME         = 'so101_car_pick_and_place'
EXPERIMENT_NAME   = '50_episodes_v0'
POLICY_CHECKPOINT = '60000'
POLICY_TYPE       = 'act'
NUM_EPISODES      = 8
FPS               = 30
EPISODE_TIME_SEC  = 60
RESET_TIME_SEC    = 5
EVAL_TYPE         = '12_test_case'

Log to HF if pushing:

In [3]:
if PUSH_TO_HUB:
    os.system(f"hf auth login --token {os.getenv('HUGGINGFACE_TOKEN')}")

In [4]:
# Resolve path or HF id
if POLICY_CHECKPOINT is None:
    POLICY_CHECKPOINT = "last"
policy_path = POLICIES_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / "checkpoints" / POLICY_CHECKPOINT / "pretrained_model"

if not policy_path.exists():
    # load from Hugging Face Hub
    policy_id = f"{HF_NAME}/{POLICY_TYPE}-{REPO_NAME}-{EXPERIMENT_NAME}"
    snapshot_download(
    repo_id                = policy_id,
    revision               = "main",
    local_dir              = str(policy_path),
    local_dir_use_symlinks = False
    )

policy_config = PreTrainedConfig.from_pretrained(policy_path)
policy_config.pretrained_path = policy_path # type: ignore
pprint.pprint(policy_config)

ACTConfig(n_obs_steps=1,
          input_features={'observation.images.top_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                                                      shape=(3,
                                                                             480,
                                                                             640)),
                          'observation.images.wrist_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>,
                                                                        shape=(3,
                                                                               480,
                                                                               640)),
                          'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>,
                                                             shape=(6,))},
          output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>,
  

In [5]:
policy_config.input_features

{'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(6,)),
 'observation.images.wrist_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 480, 640)),
 'observation.images.top_cam': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 480, 640))}

Policy Overrides:

In [19]:
print(f"Original n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")
policy_config.n_action_steps = 100
policy_config.temporal_ensemble_coeff = 0.01
print(f"Override n_action_steps = {policy_config.n_action_steps}, temporal_ensemble = {policy_config.temporal_ensemble_coeff}")

Original n_action_steps = 100, temporal_ensemble = None
Override n_action_steps = 100, temporal_ensemble = 0.01


Build Dataset:

In [20]:
dataset_path = EVAL_DIR / POLICY_TYPE / REPO_NAME / f"{EXPERIMENT_NAME}-{EVAL_TYPE}"
resume = check_resume(dataset_path)

dscfg = DatasetRecordConfig(
    repo_id                             = f"{HF_NAME}/eval_{REPO_NAME}_{EXPERIMENT_NAME}_{EVAL_TYPE}",
    single_task                         = f"eval dataset for {REPO_NAME} with policy {EXPERIMENT_NAME}, mode = {EVAL_TYPE}",
    root                                = dataset_path.__str__(),
    fps                                 = FPS,
    episode_time_s                      = EPISODE_TIME_SEC,
    reset_time_s                        = RESET_TIME_SEC,
    num_episodes                        = NUM_EPISODES,
    video                               = True,
    push_to_hub                         = PUSH_TO_HUB,
    private                             = True,
    num_image_writer_processes          = 0,
    num_image_writer_threads_per_camera = 4,
    video_encoding_batch_size           = 1,
)

rc = RecordConfig(
    robot        = robot_config,
    dataset      = dscfg,
    teleop       = None,
    policy       = policy_config,
    display_data = True,
    play_sounds  = True,
    resume       = resume
)

In [21]:
record(rc, 
    save_to_ds    = SAVE_TO_DATASET,
    reset_pose    = FOLDED_START_POSE,
    give_feedback = False,
    log_to_file   = False)

INFO 2025-10-07 10:16:47 t/record.py:422 {'dataset': {'episode_time_s': 60,
             'fps': 30,
             'num_episodes': 8,
             'num_image_writer_processes': 0,
             'num_image_writer_threads_per_camera': 4,
             'private': True,
             'push_to_hub': True,
             'rename_map': {},
             'repo_id': 'jonathm126/eval_so101_car_pick_and_place_50_episodes_v2_12_test_case',
             'reset_time_s': 5,
             'root': '/home/jonathan/Github/IDC_Project-Sim_Real_Sim/eval/act/so101_car_pick_and_place/50_episodes_v2-12_test_case',
             'single_task': 'eval dataset for so101_car_pick_and_place with '
                            'policy 50_episodes_v2, mode = 12_test_case',
             'tags': None,
             'video': True,
             'video_encoding_batch_size': 1},
 'display_data': True,
 'play_sounds': True,
 'policy': {'chunk_size': 100,
            'device': 'cuda',
            'dim_feedforward': 3200,
            'di

Loading weights from local directory


INFO 2025-10-07 10:16:50 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-Sonix_Technology_Co.__Ltd._USB2.0_CAM1_USB2.0_CAM1-video-index0) connected.
INFO 2025-10-07 10:16:52 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-046d_Logitech_BRIO_8F54E371-video-index0) connected.
INFO 2025-10-07 10:16:52 follower.py:104 so_101_follower SO101Follower connected.
INFO 2025-10-07 10:16:53 ls/utils.py:227 Recording episode 50
INFO 2025-10-07 10:17:19 ls/utils.py:227 Reset the environment


Right arrow key pressed. Exiting loop...


Map: 100%|██████████| 738/738 [00:00<00:00, 3251.80 examples/s]
Generating train split: 60268 examples [00:00, 4075556.45 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 20250110 (Red Hat 14.2.1-7)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:14:07
Svt[info]: -------------------------------------------
Svt[info]: Level of Parallelism: 5
Svt[info]: Number of PPCS 140
Svt[info]: [asm level on system : up to avx2]
Svt[info]: [asm level selected : up to avx2]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / fps numerator / fps denominator 		: 640 / 480 / 30 / 1
Svt[info]: SVT [config]: bit-depth / color format 					: 8 / YUV420
Svt[info]: SVT [config]: preset / tune / pred struct 					: 8 / PSNR / random access
Svt[info]: SVT [config]: gop size / mini-gop size / key-fr

...

Right arrow key pressed. Exiting loop...


Map: 100%|██████████| 1093/1093 [00:00<00:00, 3337.54 examples/s]
Generating train split: 68565 examples [00:00, 4493265.22 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 20250110 (Red Hat 14.2.1-7)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:14:07
Svt[info]: -------------------------------------------
Svt[info]: Level of Parallelism: 5
Svt[info]: Number of PPCS 140
Svt[info]: [asm level on system : up to avx2]
Svt[info]: [asm level selected : up to avx2]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / fps numerator / fps denominator 		: 640 / 480 / 30 / 1
Svt[info]: SVT [config]: bit-depth / color format 					: 8 / YUV420
Svt[info]: SVT [config]: preset / tune / pred struct 					: 8 / PSNR / random access
Svt[info]: SVT [config]: gop size / mini-gop size / key-

Right arrow key pressed. Exiting loop...


Map: 100%|██████████| 1029/1029 [00:00<00:00, 3265.66 examples/s]
Generating train split: 69594 examples [00:00, 4941400.20 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 20250110 (Red Hat 14.2.1-7)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:14:07
Svt[info]: -------------------------------------------
Svt[info]: Level of Parallelism: 5
Svt[info]: Number of PPCS 140
Svt[info]: [asm level on system : up to avx2]
Svt[info]: [asm level selected : up to avx2]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / fps numerator / fps denominator 		: 640 / 480 / 30 / 1
Svt[info]: SVT [config]: bit-depth / color format 					: 8 / YUV420
Svt[info]: SVT [config]: preset / tune / pred struct 					: 8 / PSNR / random access
Svt[info]: SVT [config]: gop size / mini-gop size / key-

LeRobotDataset({
    Repository ID: 'jonathm126/eval_so101_car_pick_and_place_50_episodes_v2_12_test_case',
    Number of selected episodes: '58',
    Number of selected samples: '69594',
    Features: '['action', 'observation.state', 'observation.images.wrist_cam', 'observation.images.top_cam', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',